# 01 试点数据清洗与标准化

**项目**: MGS 4701 W01 Fall 2026 — Track 5：BA/DS 硕士招生面试  
**数据**: `data/pilot_dataset_raw.csv`（100 条，手收集，来源见 `docs/collection_protocol.md`）  
**编码表**: 字段标准化规则见 `docs/coding_scheme.md`  
**产出**: `data/pilot_dataset_cleaned.csv`

本 notebook 完成：
1. 「面试形式」→ 8 类分析编码（`selective / async_video / live_video / phone / written_live / group / mixed / none`）
2. 「面试平台」→ 平台标签抽取
3. 「题目数量/时长」→ 题目数与总时长（分钟）解析
4. 数据质量校验（记录数、重复、缺失）

In [1]:
# -*- coding: utf-8 -*-
import re
import pandas as pd

RAW = "../data/pilot_dataset_raw.csv"
OUT = "../data/pilot_dataset_cleaned.csv"

df = pd.read_csv(RAW)
print(f"原始记录数: {len(df)} | 列数: {df.shape[1]}")
df.head(3)

原始记录数: 100 | 列数: 11


,序号,国家/地区,院校,项目名称,面试形式,面试平台,题目数量/时长,问题类型,典型面试问题示例,数据来源（可搜索验证）,来源类型
0,1,美国,麻省理工学院 (MIT),Master of Business Analytics (MBAn),真人视频面试（1V1）+ 视频提交,Zoom / 自建平台,视频1:60秒自我介绍；视频2:10秒准备+60秒作答1题；面试30分钟（行为+技术）,行为题+技术题（数学/概率/统计/编程）+PPT项目展示,1. Tell me about a data science project you ar...,https://mitsloan.mit.edu/master-of-business-an...,院校官网 + 知乎面经
1,2,美国,哥伦比亚大学 (Columbia),MS in Business Analytics (MSBA),异步视频面试（录播）,自建视频面试平台,3题；每题90秒准备+3分钟作答,行为题（领导力/团队/动机）,1. Tell me about a time you showed leadership\...,https://www.gsb.columbia.edu/programs/msba/adm...,院校官网 + 小红书/一亩三分地
2,3,美国,纽约大学 Stern 商学院 (NYU),MS Business Analytics,异步视频面试（录播）,Kira Talent,约8-9题；每题1分30秒准备+3分钟作答,行为题+写作题+动机题,1. EHICS讨论\n2. 东京杯子问题\n3. 15分钟即兴写作（多样性话题）,小红书/一亩三分地面经帖 + NYU Stern官网,小红书/一亩三分地 + 院校官网


## 1. 面试形式标准化

原始「面试形式」字段有 49 种自由文本写法（如「部分候选人面试（评估制）」「视频面试（Zoom）」「真人线上单面」），
先归并为 **8 类分析编码**（编码定义见 `docs/coding_scheme.md`），用于后续跨地区/跨项目比较。

In [2]:
FORMAT_MAP = {
    "部分候选人面试（评估制）": "selective", "部分候选人受邀面试": "selective",
    "部分项目需面试": "selective", "部分申请者需面试": "selective",
    "部分申请者获邀面试": "selective", "材料审核+部分面试": "selective",
    "有面试环节": "selective", "行为面试（部分候选人）": "selective",
    "行为面试（邀请制）": "selective",
    "部分项目需面试（研究型MPhil/MRes必面）": "selective",
    "邀请制面试（校园/电话/视频）": "selective", "虚拟面试（邀请制）": "selective",
    "异步视频面试（录播）": "async_video", "视频面试（录播）": "async_video",
    "案例分析视频面试": "async_video",
    "真人视频面试（1V1）+ 视频提交": "live_video",
    "真人视频面试（Zoom/Skype）": "live_video", "真人视频面试（推荐参加）": "live_video",
    "真人视频面试（Zoom）+ 技能评估": "live_video", "视频面试（部分申请者获邀）": "live_video",
    "视频面试（Zoom）": "live_video", "视频面试（Zoom）+ 编程测试": "live_video",
    "视频面试（Zoom/Skype）": "live_video", "线上Zoom面试（双教授）": "live_video",
    "全英文个面": "live_video", "真人线上单面": "live_video", "线上面试": "live_video",
    "结构化面试": "live_video", "结构化面试+案例问题": "live_video", "面试+写作": "live_video",
    "电话/Skype面试": "phone", "视频会议或电话": "live_video",
    "线上机考 + 真人面谈": "written_live", "全英文个面（笔试+面试）": "written_live",
    "笔试+面试": "written_live", "两轮面试：机面+真人面": "written_live",
    "真人单面 + 笔试": "written_live",
    "小群面": "group", "小组讨论(TBD) + 一对一面试": "group",
    "Video / 真人单面 / 群面": "mixed", "Video / 真人面 / 群面": "mixed",
    "真人线上面试（单面+群面）": "mixed",
    "无强制面试（需提交GRE/GMAT）": "none", "无强制面试（需提交GMAT/GRE）": "none",
    "无强制面试（需提交文书+推荐信）": "none", "无强制面试（可选提交补充材料）": "none",
}
FORMAT_CN = {
    "selective": "部分候选人需面试（评估制）", "async_video": "异步视频面试（录播）",
    "live_video": "真人视频面试", "phone": "电话面试", "written_live": "笔试/机考+真人面试",
    "group": "群面/小组讨论", "mixed": "混合形式", "none": "无强制面试",
}

unmapped = set(df["面试形式"]) - set(FORMAT_MAP)
print("未覆盖的原始写法数量:", len(unmapped))
print("原始写法唯一值 49 → 编码 8 类")

未覆盖的原始写法数量: 0
原始写法唯一值 49 → 编码 8 类


## 2. 平台标签与题目数量/时长解析

- **平台标签**：从「面试平台」自由文本中抽取标准化平台（Zoom / Kira Talent / WePow / 自建平台 / Skype / Lantern AI / Astronaut / 线下·校园 / 电话等），一行为多标签。
- **题目数量**：解析「约N题 / N-M题（取中值）/ Q1-Q6 / 3题×1.5分钟」等写法。
- **时长（分钟）**：N-M分钟取中值、N分钟取整；纯笔试/机考/技能评估/按小时计段不计入（如「笔试2小时+面试10-15分钟」只取 12.5 分钟）。

In [3]:
PLATFORM_PATTERNS = [
    ("Zoom", r"zoom"), ("Kira Talent", r"kira"), ("WePow", r"wepow"),
    ("自建平台", r"自建平台"), ("Skype", r"skype"), ("Lantern AI", r"lantern"),
    ("Astronaut", r"astronaut"), ("在线机考平台", r"机考|在线平台"),
    ("线下/校园", r"线下|校园"), ("电话", r"电话"),
]

def parse_platform_tags(platform_str):
    if pd.isna(platform_str):
        return ""
    text = str(platform_str).lower()
    return ",".join(tag for tag, pat in PLATFORM_PATTERNS if re.search(pat, text))

def parse_question_count(text):
    if pd.isna(text):
        return None
    s = str(text)
    m = re.search(r"约?(\d{1,2})\s*[-—–~]\s*(\d{1,2})\s*题", s)
    if m:
        return round((int(m.group(1)) + int(m.group(2))) / 2, 1)
    m = re.search(r"约?(\d{1,2})\s*题", s)
    if m:
        return int(m.group(1))
    m = re.search(r"Q(\d)[-—–]?Q?(\d)?", s, re.IGNORECASE)
    if m:
        return int(m.group(2)) if m.group(2) else int(m.group(1))
    return None

def parse_duration_min(text):
    if pd.isna(text):
        return None
    total, found = 0.0, False
    for seg in re.split(r"[；;，,+＋]", str(text)):
        if re.search(r"笔试|机考|技能评估|小时", seg) and not re.search(r"面试|真人面|面谈", seg):
            continue
        m = re.search(r"(\d{1,3})\s*[-—–~]\s*(\d{1,3})\s*分钟", seg)
        if m:
            total += (int(m.group(1)) + int(m.group(2))) / 2; found = True; continue
        m = re.search(r"约?(\d{1,3})\s*分钟", seg)
        if m:
            total += int(m.group(1)); found = True
    return round(total, 1) if found else None

In [4]:
df["面试形式编码"] = df["面试形式"].map(FORMAT_MAP)
df["面试形式分类"] = df["面试形式编码"].map(FORMAT_CN)
df["平台标签"] = df["面试平台"].apply(parse_platform_tags)
df["题目数量(解析)"] = df["题目数量/时长"].apply(parse_question_count)
df["时长分钟(解析)"] = df["题目数量/时长"].apply(parse_duration_min)

assert df["面试形式编码"].notna().all(), "存在未编码的面试形式"
df.to_csv(OUT, index=False, encoding="utf-8-sig")
print("清洗完成 →", OUT)

清洗完成 → ../data/pilot_dataset_cleaned.csv


## 3. 数据质量校验

In [5]:
print("总记录数:", len(df))
print("重复(院校+项目):", df.duplicated(subset=["院校", "项目名称"]).sum())
print("关键字段缺失:", df[["国家/地区", "院校", "项目名称", "面试形式", "问题类型"]].isna().sum().sum())
print("\n面试形式编码分布:")
print(df["面试形式编码"].value_counts().to_string())
print("\n平台标签 Top8:")
print(df["平台标签"].value_counts().head(8).to_string())
print("\n时长解析: 有值", df["时长分钟(解析)"].notna().sum(), "行 | 中位数", df["时长分钟(解析)"].median(), "分钟")

总记录数: 100
重复(院校+项目): 0
关键字段缺失: 0

面试形式编码分布:
面试形式编码
selective       43
live_video      24
async_video     14
written_live     7
none             6
mixed            3
group            2
phone            1

平台标签 Top8:
平台标签
Zoom,线下/校园          27
Zoom                23
Kira Talent          9
                     8
Zoom,自建平台            6
Zoom,电话              5
Kira Talent,自建平台     4
自建平台                 3

时长解析: 有值 64 行 | 中位数 17.5 分钟


In [6]:
df[["序号", "国家/地区", "院校", "项目名称", "面试形式编码", "面试形式分类", "平台标签", "题目数量(解析)", "时长分钟(解析)"]].head(10)

,序号,国家/地区,院校,项目名称,面试形式编码,面试形式分类,平台标签,题目数量(解析),时长分钟(解析)
0,1,美国,麻省理工学院 (MIT),Master of Business Analytics (MBAn),live_video,真人视频面试,"Zoom,自建平台",1.0,30.0
1,2,美国,哥伦比亚大学 (Columbia),MS in Business Analytics (MSBA),async_video,异步视频面试（录播）,,3.0,3.0
2,3,美国,纽约大学 Stern 商学院 (NYU),MS Business Analytics,async_video,异步视频面试（录播）,Kira Talent,8.5,3.0
3,4,美国,加州大学圣地亚哥分校 Rady 管理学院 (UCSD),MS in Business Analytics (MSBA),async_video,异步视频面试（录播）,,5.0,NaN
4,5,美国,西北大学 McCormick 工程学院 (Northwestern),M.S. in Analytics (MSiA),live_video,真人视频面试,"Zoom,Skype",NaN,22.5
5,6,美国,康奈尔大学 (Cornell),MS in Business Analytics (MSBA),live_video,真人视频面试,自建平台,NaN,12.5
6,7,美国,杜克大学 Fuqua 商学院 (Duke),MQM: Business Analytics,live_video,真人视频面试,"Zoom,Skype,线下/校园,电话",NaN,37.5
7,8,美国,南加州大学 Marshall 商学院 (USC),MS in Finance (MSF),async_video,异步视频面试（录播）,Kira Talent,5.5,NaN
8,9,美国,约翰斯·霍普金斯大学 (JHU),MS in Applied Data Science,async_video,异步视频面试（录播）,Kira Talent,5.0,1.5
9,10,美国,华盛顿大学圣路易斯分校 (WUSTL),MS in Business Analytics,async_video,异步视频面试（录播）,Kira Talent,5.5,NaN


## 小结

- 100 条记录全部完成标准化，无重复、无关键字段缺失。
- 面试形式 49 种自由文本 → 8 类编码；`selective`（部分候选人评估制）占比最高（43%），其次真人视频（24%）、异步录播视频（14%）。
- 题目数量仅在 17 条中可显式解析（其余原文未报告）；时长在 64 条中可解析，中位数约 17.5 分钟。
- 解析字段的局限（如「约N题」取中值、单题时长未乘题数）记录于 `docs/coding_scheme.md`。